# EDA Objective

The purpose of this notebook is to understand sales patterns, seasonality, product performance, event effects, SNAP days, prices, and forecasting challenges for store `CA_1`. This is exploratory analysis only; no forecasting models or lag features are created here.

## Import Libraries and Load Data

Load the transformed CA_1 Parquet dataset without modifying or overwriting it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid", context="notebook")
PRIMARY_COLOR = "#2F6F9F"
SECONDARY_COLOR = "#D98E04"
ACCENT_COLOR = "#5E8C61"

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "ca1_sales_long.parquet").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/processed/ca1_sales_long.parquet")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ca1_sales_long.parquet"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

def save_figure(fig: plt.Figure, filename: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    return output_path

def finish_figure(fig: plt.Figure, filename: str) -> Path:
    output_path = save_figure(fig, filename)
    plt.show()
    plt.close(fig)
    return output_path

In [ ]:
df = pd.read_parquet(DATA_PATH)

if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"])

load_summary = pd.DataFrame(
    [
        {"metric": "Rows", "value": df.shape[0]},
        {"metric": "Columns", "value": df.shape[1]},
        {"metric": "Minimum date", "value": df["date"].min()},
        {"metric": "Maximum date", "value": df["date"].max()},
        {"metric": "Memory usage MB", "value": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2)},
    ]
)

display(load_summary)
display(pd.DataFrame({"columns": df.columns}))

## Overall Sales Trend

Aggregate total unit sales by date to inspect broad movement over time.

In [ ]:
daily_sales = (
    df.groupby("date", as_index=False)
    .agg(total_sales=("sales", "sum"))
    .sort_values("date")
)
daily_sales["rolling_28d_sales"] = daily_sales["total_sales"].rolling(28, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_sales["date"], daily_sales["total_sales"], color=PRIMARY_COLOR, linewidth=1)
ax.set_title("CA_1 Daily Unit Sales")
ax.set_xlabel("Date")
ax.set_ylabel("Total unit sales")
fig.autofmt_xdate()
daily_sales_trend_path = finish_figure(fig, "daily_sales_trend.png")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_sales["date"], daily_sales["rolling_28d_sales"], color=SECONDARY_COLOR, linewidth=2)
ax.set_title("CA_1 Daily Unit Sales, 28-Day Rolling Average")
ax.set_xlabel("Date")
ax.set_ylabel("28-day average unit sales")
fig.autofmt_xdate()
rolling_sales_trend_path = finish_figure(fig, "daily_sales_28d_rolling_average.png")

In [ ]:
trend_summary = pd.DataFrame(
    [
        {"metric": "Average daily sales", "value": round(daily_sales["total_sales"].mean(), 2)},
        {"metric": "Lowest daily sales", "value": int(daily_sales["total_sales"].min())},
        {"metric": "Lowest sales date", "value": daily_sales.loc[daily_sales["total_sales"].idxmin(), "date"]},
        {"metric": "Highest daily sales", "value": int(daily_sales["total_sales"].max())},
        {"metric": "Highest sales date", "value": daily_sales.loc[daily_sales["total_sales"].idxmax(), "date"]},
        {"metric": "First 28-day average", "value": round(daily_sales["rolling_28d_sales"].iloc[0], 2)},
        {"metric": "Final 28-day average", "value": round(daily_sales["rolling_28d_sales"].iloc[-1], 2)},
    ]
)

display(trend_summary)

direction = "increased" if daily_sales["rolling_28d_sales"].iloc[-1] > daily_sales["rolling_28d_sales"].iloc[0] else "declined"
print(
    f"The 28-day average {direction} from the start to the end of the available period. "
    f"The largest daily peak occurs on {daily_sales.loc[daily_sales['total_sales'].idxmax(), 'date'].date()}, "
    f"and the lowest daily sales occur on {daily_sales.loc[daily_sales['total_sales'].idxmin(), 'date'].date()}."
)

## Weekly Seasonality

Compare average sales by weekday, ordered Monday through Sunday.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekly_sales = (
    daily_sales.assign(day_of_week=daily_sales["date"].dt.day_name())
    .groupby("day_of_week", as_index=False)
    .agg(avg_sales=("total_sales", "mean"))
)
weekly_sales["day_of_week"] = pd.Categorical(weekly_sales["day_of_week"], categories=weekday_order, ordered=True)
weekly_sales = weekly_sales.sort_values("day_of_week")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=weekly_sales, x="day_of_week", y="avg_sales", color=PRIMARY_COLOR, ax=ax)
ax.set_title("Average Daily Sales by Weekday")
ax.set_xlabel("Day of week")
ax.set_ylabel("Average daily unit sales")
ax.tick_params(axis="x", rotation=30)
weekly_seasonality_path = finish_figure(fig, "weekly_seasonality.png")

strongest_day = weekly_sales.loc[weekly_sales["avg_sales"].idxmax()]
weakest_day = weekly_sales.loc[weekly_sales["avg_sales"].idxmin()]
display(weekly_sales)
print(f"Strongest sales day: {strongest_day['day_of_week']} with average sales of {strongest_day['avg_sales']:.2f}.")
print(f"Weakest sales day: {weakest_day['day_of_week']} with average sales of {weakest_day['avg_sales']:.2f}.")

## Monthly and Yearly Patterns

Summarize monthly and yearly sales to identify longer seasonal patterns.

In [ ]:
monthly_sales = (
    daily_sales.assign(month=daily_sales["date"].dt.month)
    .groupby("month", as_index=False)
    .agg(avg_daily_sales=("total_sales", "mean"), total_sales=("total_sales", "sum"))
)

yearly_sales = (
    daily_sales.assign(year=daily_sales["date"].dt.year)
    .groupby("year", as_index=False)
    .agg(total_sales=("total_sales", "sum"), avg_daily_sales=("total_sales", "mean"), days=("date", "nunique"))
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=monthly_sales, x="month", y="avg_daily_sales", color=ACCENT_COLOR, ax=ax)
ax.set_title("Average Daily Sales by Calendar Month")
ax.set_xlabel("Month")
ax.set_ylabel("Average daily unit sales")
monthly_sales_path = finish_figure(fig, "monthly_sales.png")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=yearly_sales, x="year", y="total_sales", color=SECONDARY_COLOR, ax=ax)
ax.set_title("Total Sales by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Total unit sales")
yearly_sales_path = finish_figure(fig, "yearly_sales.png")

display(monthly_sales)
display(yearly_sales)

best_month = monthly_sales.loc[monthly_sales["avg_daily_sales"].idxmax()]
lowest_month = monthly_sales.loc[monthly_sales["avg_daily_sales"].idxmin()]
print(f"Highest average daily sales occur in month {int(best_month['month'])}.")
print(f"Lowest average daily sales occur in month {int(lowest_month['month'])}.")

## Category and Department Performance

Compare total and average sales by product category and department.

In [ ]:
category_sales = (
    df.groupby("cat_id", as_index=False)
    .agg(total_sales=("sales", "sum"), avg_sales=("sales", "mean"), item_count=("item_id", "nunique"))
    .sort_values("total_sales", ascending=False)
)

department_sales = (
    df.groupby("dept_id", as_index=False)
    .agg(total_sales=("sales", "sum"), avg_sales=("sales", "mean"), item_count=("item_id", "nunique"))
    .sort_values("total_sales", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=category_sales, x="cat_id", y="total_sales", color=PRIMARY_COLOR, ax=ax)
ax.set_title("Total Sales by Category")
ax.set_xlabel("Category")
ax.set_ylabel("Total unit sales")
category_sales_path = finish_figure(fig, "category_sales.png")

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=department_sales, x="dept_id", y="total_sales", color=ACCENT_COLOR, ax=ax)
ax.set_title("Total Sales by Department")
ax.set_xlabel("Department")
ax.set_ylabel("Total unit sales")
ax.tick_params(axis="x", rotation=30)
department_sales_path = finish_figure(fig, "department_sales.png")

display(category_sales)
display(department_sales)

print(f"Highest category by total sales: {category_sales.iloc[0]['cat_id']}.")
print(f"Lowest category by total sales: {category_sales.iloc[-1]['cat_id']}.")
print(f"Highest department by total sales: {department_sales.iloc[0]['dept_id']}.")
print(f"Lowest department by total sales: {department_sales.iloc[-1]['dept_id']}.")

## Event Analysis

Missing event values are expected and mean no special event occurred. Compare event and non-event days without treating missing event names as quality errors.

In [ ]:
event_daily_sales = daily_sales.merge(
    df[["date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"]].drop_duplicates("date"),
    on="date",
    how="left",
)
event_daily_sales["event_day"] = event_daily_sales[["event_name_1", "event_name_2"]].notna().any(axis=1)

event_vs_none = (
    event_daily_sales.groupby("event_day", as_index=False)
    .agg(avg_daily_sales=("total_sales", "mean"), days=("date", "nunique"))
)
event_vs_none["event_day_label"] = np.where(event_vs_none["event_day"], "Event day", "Non-event day")

event_type_frames = []
for event_type_column in ["event_type_1", "event_type_2"]:
    temp = event_daily_sales.loc[event_daily_sales[event_type_column].notna(), ["date", "total_sales", event_type_column]].copy()
    temp = temp.rename(columns={event_type_column: "event_type"})
    event_type_frames.append(temp)

event_type_sales = pd.concat(event_type_frames, ignore_index=True) if event_type_frames else pd.DataFrame(columns=["date", "total_sales", "event_type"])
event_type_summary = (
    event_type_sales.groupby("event_type", as_index=False)
    .agg(avg_daily_sales=("total_sales", "mean"), days=("date", "nunique"))
    .query("days >= 3")
    .sort_values("avg_daily_sales", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=event_vs_none, x="event_day_label", y="avg_daily_sales", color=PRIMARY_COLOR, ax=axes[0])
axes[0].set_title("Average Sales: Event vs Non-Event Days")
axes[0].set_xlabel("")
axes[0].set_ylabel("Average daily unit sales")

if event_type_summary.empty:
    axes[1].text(0.5, 0.5, "No event type has at least 3 observations", ha="center", va="center")
    axes[1].set_axis_off()
else:
    sns.barplot(data=event_type_summary, x="event_type", y="avg_daily_sales", color=SECONDARY_COLOR, ax=axes[1])
    axes[1].set_title("Average Sales by Event Type")
    axes[1].set_xlabel("Event type")
    axes[1].set_ylabel("Average daily unit sales")
    axes[1].tick_params(axis="x", rotation=30)

event_sales_path = finish_figure(fig, "event_sales.png")

display(event_vs_none)
display(event_type_summary)

if len(event_vs_none) == 2:
    event_avg = event_vs_none.loc[event_vs_none["event_day"], "avg_daily_sales"].iloc[0]
    non_event_avg = event_vs_none.loc[~event_vs_none["event_day"], "avg_daily_sales"].iloc[0]
    relation = "higher" if event_avg > non_event_avg else "lower"
    print(f"Average sales on event days are {relation} than non-event days in this CA_1 sample.")

## SNAP Analysis

Compare average daily sales when California SNAP is active versus inactive.

In [ ]:
snap_daily_sales = daily_sales.merge(
    df[["date", "snap_CA"]].drop_duplicates("date"),
    on="date",
    how="left",
)
snap_summary = (
    snap_daily_sales.groupby("snap_CA", as_index=False)
    .agg(avg_daily_sales=("total_sales", "mean"), days=("date", "nunique"))
)
snap_summary["snap_status"] = np.where(snap_summary["snap_CA"].eq(1), "SNAP active", "SNAP inactive")

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=snap_summary, x="snap_status", y="avg_daily_sales", color=PRIMARY_COLOR, ax=ax)
ax.set_title("Average Daily Sales by SNAP Status")
ax.set_xlabel("")
ax.set_ylabel("Average daily unit sales")
snap_sales_path = finish_figure(fig, "snap_sales.png")

display(snap_summary)
if snap_summary["snap_CA"].nunique() == 2:
    active = snap_summary.loc[snap_summary["snap_CA"].eq(1), "avg_daily_sales"].iloc[0]
    inactive = snap_summary.loc[snap_summary["snap_CA"].eq(0), "avg_daily_sales"].iloc[0]
    relation = "higher" if active > inactive else "lower"
    print(f"Average daily sales are {relation} on SNAP-active days than on SNAP-inactive days.")

## Price Analysis

Inspect price distribution and sales-price associations. These are associations only, not causal claims.

In [ ]:
price_data = df.loc[df["sell_price"].notna()].copy()

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(price_data["sell_price"], bins=50, color=PRIMARY_COLOR, ax=ax)
ax.set_title("Sell Price Distribution")
ax.set_xlabel("Sell price")
ax.set_ylabel("Record count")
price_distribution_path = finish_figure(fig, "price_distribution.png")

price_stats_by_category = (
    price_data.groupby("cat_id", as_index=False)
    .agg(
        min_price=("sell_price", "min"),
        median_price=("sell_price", "median"),
        mean_price=("sell_price", "mean"),
        max_price=("sell_price", "max"),
        avg_sales=("sales", "mean"),
    )
    .sort_values("mean_price", ascending=False)
)

item_price_sales = (
    price_data.groupby(["item_id", "cat_id", "dept_id"], as_index=False)
    .agg(avg_price=("sell_price", "mean"), avg_sales=("sales", "mean"), total_sales=("sales", "sum"))
)
scatter_sample = item_price_sales.sample(n=min(1000, len(item_price_sales)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=scatter_sample, x="avg_price", y="avg_sales", hue="cat_id", alpha=0.7, ax=ax)
ax.set_title("Average Price vs Average Sales by Item")
ax.set_xlabel("Average sell price")
ax.set_ylabel("Average unit sales")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
price_sales_scatter_path = finish_figure(fig, "price_sales_scatter.png")

display(price_stats_by_category)
display(item_price_sales[["avg_price", "avg_sales"]].corr())
print("The scatter plot describes association between item-level average price and sales, not causation.")

## Zero-Sales and Intermittent Demand

Estimate how often item-day records have zero sales. Intermittent demand makes forecasting difficult because many item series have long quiet periods and occasional spikes.

In [ ]:
zero_sales_pct = (df["sales"].eq(0).mean() * 100).round(2)

zero_by_category = (
    df.groupby("cat_id")
    .agg(zero_sales_pct=("sales", lambda s: round(s.eq(0).mean() * 100, 2)), records=("sales", "size"))
    .reset_index()
    .sort_values("zero_sales_pct", ascending=False)
)

zero_by_department = (
    df.groupby("dept_id")
    .agg(zero_sales_pct=("sales", lambda s: round(s.eq(0).mean() * 100, 2)), records=("sales", "size"))
    .reset_index()
    .sort_values("zero_sales_pct", ascending=False)
)

display(pd.DataFrame([{"metric": "Overall zero-sales percentage", "value": zero_sales_pct}]))
display(zero_by_category)
display(zero_by_department)
print(f"{zero_sales_pct}% of CA_1 item-day records have zero sales, which signals intermittent demand.")

## Top Products

Identify the top 10 items by total unit sales and show their category and department.

In [ ]:
top_items = (
    df.groupby(["item_id", "cat_id", "dept_id"], as_index=False)
    .agg(total_sales=("sales", "sum"), avg_sales=("sales", "mean"))
    .sort_values("total_sales", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_items.sort_values("total_sales"), y="item_id", x="total_sales", color=PRIMARY_COLOR, ax=ax)
ax.set_title("Top 10 Items by Total Sales")
ax.set_xlabel("Total unit sales")
ax.set_ylabel("Item")
top_items_path = finish_figure(fig, "top_items.png")

display(top_items)
print(f"Top-selling item: {top_items.iloc[0]['item_id']} from {top_items.iloc[0]['cat_id']} / {top_items.iloc[0]['dept_id']}.")

## Forecasting Implications

Summarize the EDA patterns that should guide later feature engineering and model development.

In [ ]:
implications = pd.DataFrame(
    [
        {"area": "Trend", "implication": "Use time index and rolling summaries later to capture changing demand levels."},
        {"area": "Weekly seasonality", "implication": "Add weekday and calendar-cycle features later."},
        {"area": "Category differences", "implication": "Modeling may need category or department identifiers because demand levels differ by group."},
        {"area": "Event and SNAP patterns", "implication": "Use event flags and SNAP indicators as candidate future features."},
        {"area": "Price relationships", "implication": "Use price and price-availability fields later, while avoiding causal claims from EDA alone."},
        {"area": "Zero-sales behavior", "implication": "Intermittent demand suggests item-level forecasts may need methods robust to many zeros."},
    ]
)

implications

## Save Figures

Important figures are saved as PNG files in `outputs/figures/` as each plotting cell runs.

In [ ]:
saved_figures = pd.DataFrame(
    [
        {"figure": "Daily sales trend", "path": daily_sales_trend_path},
        {"figure": "Daily sales 28-day rolling average", "path": rolling_sales_trend_path},
        {"figure": "Weekly seasonality", "path": weekly_seasonality_path},
        {"figure": "Monthly sales", "path": monthly_sales_path},
        {"figure": "Yearly sales", "path": yearly_sales_path},
        {"figure": "Category sales", "path": category_sales_path},
        {"figure": "Department sales", "path": department_sales_path},
        {"figure": "Event sales", "path": event_sales_path},
        {"figure": "SNAP sales", "path": snap_sales_path},
        {"figure": "Price distribution", "path": price_distribution_path},
        {"figure": "Price-sales scatter", "path": price_sales_scatter_path},
        {"figure": "Top items", "path": top_items_path},
    ]
)

saved_figures